# Local RAG Notebook

This notebook builds a small Retrieval-Augmented Generation style pipeline over the local files in `../data`. It uses local retrieval only, so it does not require an API key.

In [ ]:
from pathlib import Path
from textwrap import shorten
import re

import numpy as np
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def has_supported_files(path: Path) -> bool:
    supported_suffixes = {".txt", ".md", ".html", ".pdf"}
    return path.exists() and any(
        item.is_file() and item.suffix.lower() in supported_suffixes
        for item in path.rglob("*")
    )


def find_data_dir() -> Path:
    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd() / ".vscode" / "data",
    ]
    for candidate in candidates:
        if has_supported_files(candidate):
            return candidate
    raise FileNotFoundError("Could not find the local data directory")


DATA_DIR = find_data_dir()
DATA_DIR

## Load Documents

In [ ]:
def clean_text(text: str) -> str:
    """Normalize whitespace while preserving readable text."""
    return re.sub(r"\s+", " ", text or "").strip()


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf_file(path: Path) -> str:
    reader = PdfReader(str(path))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)


def load_documents(data_dir: Path) -> list[dict[str, str]]:
    loaders = {
        ".txt": read_text_file,
        ".md": read_text_file,
        ".html": read_text_file,
        ".pdf": read_pdf_file,
    }
    documents = []

    for path in sorted(data_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in loaders:
            continue

        text = clean_text(loaders[path.suffix.lower()](path))
        if text:
            documents.append({"source": str(path.relative_to(data_dir)), "text": text})

    return documents


documents = load_documents(DATA_DIR)
print(f"Loaded {len(documents)} documents from {DATA_DIR}")
for document in documents:
    print(f"- {document['source']}: {shorten(document['text'], width=100)}")

## Chunk Text

In [ ]:
def chunk_text(text: str, chunk_size: int = 450, overlap: int = 80) -> list[str]:
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


chunks = []
for document in documents:
    for index, chunk in enumerate(chunk_text(document["text"])):
        chunks.append({"source": document["source"], "chunk_id": index, "text": chunk})

print(f"Created {len(chunks)} chunks")
chunks[:2]

## Build Retriever

In [ ]:
if not chunks:
    raise ValueError(f"No readable documents found in {DATA_DIR}")

vectorizer = TfidfVectorizer(stop_words="english")
chunk_matrix = vectorizer.fit_transform([chunk["text"] for chunk in chunks])


def retrieve(query: str, top_k: int = 3) -> list[dict[str, object]]:
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, chunk_index in enumerate(top_indices, start=1):
        chunk = chunks[int(chunk_index)]
        results.append({
            "rank": rank,
            "score": float(scores[chunk_index]),
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
        })
    return results


retrieve("What is Python used for?")

## Ask Questions

In [ ]:
def answer_question(query: str, top_k: int = 3) -> str:
    matches = retrieve(query, top_k=top_k)
    useful_matches = [match for match in matches if match["score"] > 0]

    if not useful_matches:
        return "I could not find relevant information in the local documents."

    lines = [f"Question: {query}", "", "Most relevant local passages:"]
    for match in useful_matches:
        lines.append(
            f"\n{match['rank']}. {match['source']} "
            f"(chunk {match['chunk_id']}, score {match['score']:.3f})\n"
            f"{match['text']}"
        )
    return "\n".join(lines)


print(answer_question("Why is Python useful for RAG applications?"))